## Week 5 Configuration
Run these cells once before the ETL sections to configure Spark, MinIO (S3A), H3, and Cassandra.

In [2]:
import os, sys, subprocess, pkgutil
#pkgutil → checks if package exists
def ensure_pkg(name):
    if pkgutil.find_loader(name) is None:
        print(f"Installing {name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", name])
    else:
        print(f"OK: {name} already installed")

for pkg in ["h3"]:
    ensure_pkg(pkg)

MINIO_ENDPOINT = "http://localhost:9000"
MINIO_ACCESS = "admin"
MINIO_SECRET = "password123"
MINIO_REGION = "us-east-1"

RAW_PORTO = "s3a://raw/porto-trips/"
RAW_NYC = "s3a://raw/nyc-tlc/"
CURATED_PORTO = "s3a://curated/porto-trips/"
CURATED_NYC = "s3a://curated/nyc-demand/"

CASSANDRA_HOST = "localhost"
H3_RESOLUTION = 8

OK: h3 already installed


### Pourquoi utilise-t-on H3 ?

Travailler avec des coordonnées brutes (latitude/longitude) peut être lent et difficile à analyser.  
H3, créé par Uber, permet de diviser la carte en hexagones afin d’optimiser le traitement géospatial.
customer at lat=34.02, lon=-6.83

We say:

customer is in H3 cell 8928308280fffff

#### Avantages
- regrouper les points proches
- analyser les zones facilement
- calculer les tendances par région
- améliorer les performances des requêtes
- créer des heatmaps et du clustering

#### Pourquoi des hexagones ?
Les hexagones sont meilleurs que les carrés car ils offrent :
- une distance plus uniforme entre voisins
- moins de distorsion
- une analyse géographique plus fluide

#### Que signifie `resolution = 8` ?
La résolution représente la taille des hexagones :
- faible résolution → grands hexagones
- haute résolution → petits hexagones détaillés

Low resolution = FAST
Why?
Fewer hexagons overall
Less data to store
Faster grouping and aggregation

`Resolution = 8` correspond généralement à un niveau quartier/local.

In [3]:
import os
import sys
import pyspark

# Active Config (MyVersion style)
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['SPARK_HOME'] = '/usr/local/spark'  # where spark-submit actually lives
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ['PATH']

preferred_python = r"C:\Users\nmira\AppData\Local\Programs\Python\Python311\python.exe"
python_path = preferred_python if os.path.exists(preferred_python) else sys.executable
os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

print("Environment configured for Spark initialization.")

Environment configured for Spark initialization.


In [3]:
print(os.environ.get("SPARK_HOME"))

/usr/local/spark


In [4]:
print(os.path.exists(r"/opt/java/openjdk/bin/java"))

False


In [5]:
print(os.path.exists(r"/usr/lib/jvm/java-17-openjdk-amd64/bin/java"))

True


In [6]:
import subprocess
print(subprocess.run(['which', 'java'], capture_output=True, text=True).stdout)
print(subprocess.run(['find', '/', '-name', 'java', '-type', 'f'], capture_output=True, text=True).stdout)

/usr/lib/jvm/java-17-openjdk-amd64/bin/java

/var/lib/dpkg/alternatives/java
/usr/lib/jvm/java-17-openjdk-amd64/bin/java



### 🚀 SparkSession : Référence Rapide

**Définition :** Le point d'entrée principal et unifié pour Apache Spark (v2.0+) utilisé pour traiter les données distribuées et gérer les ressources du cluster.

**Ce qu'il fait :**
*   **Connecte** votre code d'application au cluster Spark.
*   **Lit/Écrit** les données sous plusieurs formats (CSV, Parquet, Kafka) pour générer des DataFrames.
*   **Exécute** des requêtes Spark SQL distribuées directement sur les vues et les tables.
*   **Gère** les configurations d'exécution de l'application.

**Cas d'utilisation principaux :** Création de pipelines ETL, analyse Big Data distribuée et préparation de données à grande échelle.

**Alternatives héritées (désormais fusionnées) :**
*Avant Spark 2.0, les développeurs utilisaient des contextes fragmentés. Ils sont aujourd'hui encapsulés par la `SparkSession`.*

| Contexte Hérité | Ancien Objectif | Statut dans Spark Moderne |
| :--- | :--- | :--- |
| **`SparkContext`** | Manipulation bas niveau des RDD. | Existe toujours, mais intégré dans la `SparkSession` (`spark.sparkContext`). |
| **`SQLContext`** | SQL de base et création de DataFrames. | **Obsolète**. Totalement remplacé par la `SparkSession`. |
| **`HiveContext`** | Exécution de HiveQL et lecture de tables Hive. | **Fusionné**. Accessible via `SparkSession.builder.enableHiveSupport()`. |

In [7]:
import subprocess
result = subprocess.run(['find', '/', '-name', 'spark-submit', '-type', 'f'], 
                      capture_output=True, text=True)
print(result.stdout)

/usr/local/spark-3.5.0-bin-hadoop3/bin/spark-submit
/home/jovyan/work/.venv/bin/spark-submit
/home/jovyan/work/.venv/lib/python3.12/site-packages/pyspark/bin/spark-submit



In [4]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['SPARK_HOME'] = '/usr/local/spark-3.5.0-bin-hadoop3'
os.environ['PATH'] = '/usr/lib/jvm/java-17-openjdk-amd64/bin:' + os.environ['PATH']

from pyspark.sql import SparkSession

JARS = ",".join([
    "/home/jovyan/work/storage/jars/spark-cassandra-connector-assembly_2.12-3.5.0.jar",  # ← assembly
    "/home/jovyan/work/storage/jars/hadoop-aws-3.3.4.jar",
    "/home/jovyan/work/storage/jars/aws-java-sdk-bundle-1.12.262.jar",
])

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("TaaSim") \
    .config("spark.jars", JARS) \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .config("spark.cassandra.connection.host", "cassandra") \
    .config("spark.cassandra.connection.port", "9042") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Connected!", spark.version)
print("JARs:", spark.sparkContext.getConf().get("spark.jars"))

Connected! 3.5.0
JARs: /home/jovyan/work/storage/jars/spark-cassandra-connector-assembly_2.12-3.5.0.jar,/home/jovyan/work/storage/jars/hadoop-aws-3.3.4.jar,/home/jovyan/work/storage/jars/aws-java-sdk-bundle-1.12.262.jar


In [6]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 649.7 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 6.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 2.0 MB/s eta 0:00:00a 0:00:01


## Week 5 — Spark ETL & Analytics
#### Section 1 — Spark Session connected to MinIO (S3A)


In [7]:
import boto3

s3 = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password123'
)

def minio_ls(bucket):
    response = s3.list_objects_v2(Bucket=bucket)
    print(f"\nMinIO {bucket}:")
    if 'Contents' in response:
        for obj in response['Contents']:
            print(f"  {obj['Key']}")
    else:
        print("  (empty)")

minio_ls("raw")
minio_ls("curated")
minio_ls("mls")


MinIO raw:
  nyc-tlc/yellow_tripdata_2019-01.parquet
  nyc-tlc/yellow_tripdata_2019-02.parquet
  nyc-tlc/yellow_tripdata_2019-03.parquet
  porto-trips/casablanca_real_roads_final.csv

MinIO curated:
  (empty)

MinIO mls:
  (empty)


### Section 2 — Porto ETL

Pipeline :
1. Lire les trajets depuis le fichier CSV
2. Extraire latitude/longitude depuis CASA_POLYLINE
3. Générer les cellules H3
4. Nettoyer et dédupliquer les données
5. Sauvegarder les données en Parquet

In [8]:
#it uses DATAFRAME API CODE
#The DataFrame API relies on programmatic, object-oriented methods chained together
porto_raw = spark.read.option("header", "true").csv("/home/jovyan/work/storage/data/raw/casablanca_real_roads_final.csv")
print(f"Row count  : {porto_raw.count():,}")
print(f"Columns    : {porto_raw.columns}")
porto_raw.show(3, truncate=80)

Row count  : 3,555
Columns    : ['TRIP_ID', 'TAXI_ID', 'TIMESTAMP', 'CASA_POLYLINE']
+-------------------+--------+----------+--------------------------------------------------------------------------------+
|            TRIP_ID| TAXI_ID| TIMESTAMP|                                                                   CASA_POLYLINE|
+-------------------+--------+----------+--------------------------------------------------------------------------------+
|1372636858620000589|20000589|1372636858|[[-7.5938433, 33.4837734], [-7.5923185, 33.4831759], [-7.5923029, 33.4831319]...|
|1372637303620000596|20000596|1372637303|[[-7.6848413, 33.5656518], [-7.6847662, 33.565406], [-7.6848045, 33.5654062],...|
|1372636951620000320|20000320|1372636951|[[-7.5712232, 33.4801193], [-7.5714377, 33.4801287], [-7.5769293, 33.4803416]...|
+-------------------+--------+----------+--------------------------------------------------------------------------------+
only showing top 3 rows



In [9]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType
import h3, json

# Func our extraires lat et long depuis casa polyline ─────────────────────────────────
def parse_polyline(poly_str):
    """Returns (lat, lon, duration) from CASA_POLYLINE string"""
    try:
        coords = json.loads(poly_str)
        if coords and len(coords) > 0:
            lon, lat = coords[0][0], coords[0][1]
            return (float(lat), float(lon))
    except:
        pass
    return (None, None)

# We still need ONE Python UDF for polyline parsing (no native Spark alternative)
# but we materialize (cache) BEFORE the h3 step so the UDF runs only once
from pyspark.sql.types import StructType, StructField
#creation d udf spark pour apliquer casa polyline sur cha
parse_udf = F.udf(
    lambda s: parse_polyline(s),
    StructType([
        StructField("lat", DoubleType()),
        StructField("lon", DoubleType()),
    ])
)
# Nettoyage et transformation des données
porto_parsed = (
    porto_raw
    .withColumn("coords", parse_udf(F.col("CASA_POLYLINE")))
    .withColumn("latitude",  F.col("coords.lat"))
    .withColumn("longitude", F.col("coords.lon"))
    .withColumn("duration_sec",
        F.col("TIMESTAMP").cast("long")
    )
    .withColumn("trip_start",
        F.to_timestamp(F.col("TIMESTAMP").cast("long"))
    )
      # Extraire année-mois
    .withColumn("year_month", F.date_format("trip_start", "yyyy-MM"))
    .filter(F.col("latitude").isNotNull())
    # Supprimer colonnes inutiles
    .drop("coords", "CASA_POLYLINE")
    .dropDuplicates(["TRIP_ID"])
)

#stocker les donnees en memoire pour evite rde recalculer UDF 
porto_parsed = porto_parsed.cache()
porto_parsed.count()  # triggers the UDF execution NOW, result stored in memory

#conversion H3
import pandas as pd
#convertir SparkDataFrame vers pandas por le calcul de h3 cellule locallement 
pdf = porto_parsed.select(
    "TRIP_ID",
    "latitude",
    "longitude"
).toPandas()

# Générer cellule H3 pour chaque trajet
pdf["h3_cell"] = pdf.apply(
    lambda r: h3.latlng_to_cell(r["latitude"], r["longitude"], H3_RESOLUTION)
    if pd.notna(r["latitude"]) else None,
    axis=1
)
#Retour vers Spark
h3_spark = spark.createDataFrame(pdf[["TRIP_ID", "h3_cell"]])

porto_clean = (
    porto_parsed
     # Ajouter les cellules H3
    .join(h3_spark, on="TRIP_ID", how="left")
    # Sélectionner et renommer les colonnes
    .select(
        F.col("TRIP_ID").alias("trip_id"),
        F.col("TAXI_ID").alias("taxi_id"),
        F.col("trip_start"),
        F.col("latitude"),
        F.col("longitude"),
        F.col("duration_sec"),
        F.col("h3_cell"),
        F.col("year_month"),
    )
)

print(f"Clean row count: {porto_clean.count():,}")
porto_clean.show(5)

Clean row count: 3,554
+-------------------+--------+-------------------+----------+----------+------------+---------------+----------+
|            trip_id| taxi_id|         trip_start|  latitude| longitude|duration_sec|        h3_cell|year_month|
+-------------------+--------+-------------------+----------+----------+------------+---------------+----------+
|1372636853620000380|20000380|2013-07-01 00:00:53|33.4825333|-7.5595965|  1372636853|8839aa14c1fffff|   2013-07|
|1372636858620000589|20000589|2013-07-01 00:00:58|33.4837734|-7.5938433|  1372636858|8839aa1419fffff|   2013-07|
|1372636875620000233|20000233|2013-07-01 00:01:15|33.5130808|-7.5998756|  1372636875|8839aa16e1fffff|   2013-07|
|1372636896620000360|20000360|2013-07-01 00:01:36|33.5053573|-7.5882214|  1372636896|8839aa16a9fffff|   2013-07|
|1372636951620000320|20000320|2013-07-01 00:02:31|33.4801193|-7.5712232|  1372636951|8839aa14cdfffff|   2013-07|
+-------------------+--------+-------------------+----------+----------+-

In [17]:
print(spark.sparkContext.getConf().get("spark.jars"))

/home/jovyan/work/storage/jars/spark-cassandra-connector-assembly_2.12-3.5.0.jar,/home/jovyan/work/storage/jars/hadoop-aws-3.3.4.jar,/home/jovyan/work/storage/jars/aws-java-sdk-bundle-1.12.262.jar


Maps every trip to a specific Casablanca neighborhood (Zone 1-16) using bounding box coordinates. 

It filters out any trips outside the city limits, then writes the cleaned data to MinIO (s3a://curated/porto-trips/) as Parquet files, partitioned by Month and Year for fast reading.

In [10]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

CASABLANCA_ZONES = {
    1:  (33.58, 33.62, -7.66, -7.62),
    2:  (33.58, 33.61, -7.62, -7.58),
    3:  (33.60, 33.64, -7.58, -7.54),
    4:  (33.64, 33.70, -7.54, -7.48),
    5:  (33.60, 33.65, -7.56, -7.50),
    6:  (33.57, 33.60, -7.62, -7.58),
    7:  (33.57, 33.60, -7.64, -7.60),
    8:  (33.55, 33.59, -7.58, -7.52),
    9:  (33.56, 33.59, -7.56, -7.50),
    10: (33.55, 33.58, -7.52, -7.46),
    11: (33.54, 33.58, -7.60, -7.54),
    12: (33.54, 33.58, -7.66, -7.60),
    13: (33.56, 33.60, -7.70, -7.66),
    14: (33.58, 33.62, -7.70, -7.66),
    15: (33.52, 33.58, -7.74, -7.66),
    16: (33.62, 33.68, -7.52, -7.46),
}

# Build a native Spark CASE WHEN expression — no Python UDF needed
zone_expr = F.lit(99)  # default = outside Casablanca
for zone_id, (lat_min, lat_max, lon_min, lon_max) in reversed(CASABLANCA_ZONES.items()):
    zone_expr = F.when(
        F.col("latitude").between(lat_min, lat_max) &
        F.col("longitude").between(lon_min, lon_max),
        F.lit(zone_id)
    ).otherwise(zone_expr)

porto_final = (
    porto_clean
    .withColumn("zone_id", zone_expr)
    .filter(F.col("zone_id") != 99)
    .select(
        "trip_id", "taxi_id", "trip_start",
        "latitude", "longitude", "duration_sec", "h3_cell",
        "zone_id", "year_month"
    )
)

print(f"Final row count (inside Casablanca zones): {porto_final.count():,}")

(
    porto_final
    .write
    .mode("overwrite")
    .partitionBy("year_month")
    .parquet(CURATED_PORTO)
)
print(f"✅ Porto ETL complete — written to {CURATED_PORTO}")

Final row count (inside Casablanca zones): 1,554
✅ Porto ETL complete — written to s3a://curated/porto-trips/


## Section 3 — NYC TLC ETL

The NYC dataset is already Parquet (no CSV parsing needed).
Goal: read 3 months, compute per-zone-per-hour demand counts, write to curated/.
We don't remap NYC coordinates to Casablanca — we use the TLC location IDs directly
and aggregate them. This data feeds the ML feature engineering in Week 6.

In [11]:
#read the raw NYC Taxi Dataset
nyc_raw = spark.read.parquet(RAW_NYC)

print(f"NYC row count : {nyc_raw.count():,}")
print(f"Columns       : {nyc_raw.columns}")
nyc_raw.show(3)

NYC row count : 22,612,607
Columns       : ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+-----

Cleans the NYC data (calculates exact dropoff times, removes negative distances, drops nulls) and Aggregates the data. 

Instead of keeping every single trip, it counts total trips per zone, per hour, per day. This drastically shrinks the data size before writing it to MinIO (s3a://curated/nyc-demand/). 

: you are moving from Raw Transactional Data (every single taxi ride) to Aggregated Time-Series Data (hourly demand per zone). This is exactly what Machine Learning models need to predict future demand.

Dimensionality Reduction (groupBy.agg): Instead of storing millions of individual rides, you compress the data into hourly statistical summaries. This shrinks your dataset size by orders of magnitude, making it lightweight for MinIO storage and incredibly fast for model training.

In [12]:
from pyspark.sql import functions as F

# 1. Clean and Transform the Raw NYC Data
nyc_agg = (
    nyc_raw
    # Drop rows missing critical spatial or temporal data
    .dropna(subset=["tpep_pickup_datetime", "PULocationID", "trip_distance"])
    
    # Extract temporal features needed for Machine Learning (Seasonality/Trends)
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) 
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) 
    .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("year_month", F.date_format("tpep_pickup_datetime", "yyyy-MM"))
    
    # Calculate exact trip duration in minutes using UNIX timestamps
    .withColumn(
        "duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    )
    
    # 2. Apply Business Rules / Data Quality Filters
    # Keep only logical trips: 1 min to 3 hours long, with actual distance covered
    .filter(F.col("duration_min").between(1, 180))
    .filter(F.col("trip_distance") > 0)
    
    # 3. Aggregate into Time-Series Feature Set
    # Compress millions of rows into hourly summaries per Pickup Location
    .groupBy("PULocationID", "pickup_date", "pickup_hour", "day_of_week", "year_month")
    .agg(
        F.count("*").alias("trip_count"),                  # Target variable for demand prediction
        F.avg("trip_distance").alias("avg_distance_miles"),
        F.avg("duration_min").alias("avg_duration_min"),
        F.avg("fare_amount").alias("avg_fare")
    )
)

# Trigger Action to calculate the new aggregated footprint
print(f"Aggregated rows: {nyc_agg.count():,}")
nyc_agg.show(5)

# write aggregated features to MinIO DATA LAKE
(
    nyc_agg
    .write
    .mode("overwrite")
    .partitionBy("year_month")
    .parquet(CURATED_NYC)
)
print(f"✅ NYC ETL complete — written to {CURATED_NYC}")

Aggregated rows: 296,749
+------------+-----------+-----------+-----------+----------+----------+------------------+------------------+------------------+
|PULocationID|pickup_date|pickup_hour|day_of_week|year_month|trip_count|avg_distance_miles|  avg_duration_min|          avg_fare|
+------------+-----------+-----------+-----------+----------+----------+------------------+------------------+------------------+
|         228| 2019-01-01|          0|          3|   2019-01|         4|13.407499999999999| 38.37500000000001|             41.75|
|         170| 2019-01-01|          1|          3|   2019-01|       566|  2.65565371024735|12.589428739693753|11.500883392226148|
|         114| 2018-12-31|         14|          2|   2018-12|         1|              2.69|14.516666666666667|              11.5|
|         193| 2019-01-01|          3|          3|   2019-01|         7| 6.687142857142858|17.485714285714284|21.285714285714285|
|         241| 2019-01-01|          4|          3|   2019-01|    

Section 4 — Spark SQL KPIs + Cassandra

## Section 4 — Spark SQL KPIs

Now we query the cleaned Porto data (in curated/) to compute the 4 weekly KPIs:
1. Trips per zone
2. Average trip duration per zone
3. Peak demand hours (top 3 hours by trip count)
4. Coverage gap — zones with demand but fewer than 2 active vehicles

In [13]:
from pyspark.sql import Window
from pyspark.sql import functions as F

porto_curated = spark.read.parquet(CURATED_PORTO)
porto_curated.createOrReplaceTempView("trips")

# ── KPI 1: Trips per zone (weekly) ─────────────────────────────────────────────
kpi_trips_per_zone = spark.sql("""
    SELECT
        date_trunc('week', trip_start) AS week_start,
        zone_id,
        COUNT(*) AS total_trips,
        COUNT(DISTINCT taxi_id) AS unique_taxis
    FROM trips
    GROUP BY date_trunc('week', trip_start), zone_id
    ORDER BY week_start DESC, total_trips DESC
""")
print("── KPI 1: Trips per zone (weekly) ──")
kpi_trips_per_zone.show(16)

# ── KPI 2: Average trip duration per zone (weekly) ────────────────────────────
kpi_avg_duration = spark.sql("""
    SELECT
        date_trunc('week', trip_start) AS week_start,
        zone_id,
        ROUND(AVG(duration_sec) / 60, 1) AS avg_duration_min
    FROM trips
    WHERE duration_sec > 0
    GROUP BY date_trunc('week', trip_start), zone_id
    ORDER BY week_start DESC, zone_id
""")
print("── KPI 2: Avg duration per zone (weekly) ──")
kpi_avg_duration.show(16)

# ── KPI 3: Peak demand hours (top 3 per week) ─────────────────────────────────
kpi_peak_base = spark.sql("""
    SELECT
        date_trunc('week', trip_start) AS week_start,
        HOUR(trip_start) AS hour_of_day,
        COUNT(*) AS trip_count
    FROM trips
    GROUP BY date_trunc('week', trip_start), HOUR(trip_start)
""")
rank_window = Window.partitionBy("week_start").orderBy(F.desc("trip_count"))
kpi_peak_hours = (
    kpi_peak_base
    .withColumn("rank", F.row_number().over(rank_window))
    .filter(F.col("rank") <= 3)
    .orderBy("week_start", F.desc("trip_count"))
)
print("── KPI 3: Peak demand hours (top 3 per week) ──")
kpi_peak_hours.show(30)

# ── KPI 4: Coverage gap (zones with demand but < 2 vehicles) ─────────────────
kpi_coverage_gap = (
    kpi_trips_per_zone
    .filter(F.col("unique_taxis") < 2)
    .orderBy(F.desc("total_trips"))
)
print("── KPI 4: Coverage gap (weekly) ──")
kpi_coverage_gap.show(16)

── KPI 1: Trips per zone (weekly) ──
+-------------------+-------+-----------+------------+
|         week_start|zone_id|total_trips|unique_taxis|
+-------------------+-------+-----------+------------+
|2013-07-01 00:00:00|     15|        305|         154|
|2013-07-01 00:00:00|     12|        296|         161|
|2013-07-01 00:00:00|     13|        188|         103|
|2013-07-01 00:00:00|      8|        171|         106|
|2013-07-01 00:00:00|      1|        119|          78|
|2013-07-01 00:00:00|      4|         79|          50|
|2013-07-01 00:00:00|      5|         73|          54|
|2013-07-01 00:00:00|      3|         68|          50|
|2013-07-01 00:00:00|     11|         59|          48|
|2013-07-01 00:00:00|     10|         54|          43|
|2013-07-01 00:00:00|      7|         54|          43|
|2013-07-01 00:00:00|      2|         30|          24|
|2013-07-01 00:00:00|      6|         28|          23|
|2013-07-01 00:00:00|      9|         21|          17|
|2013-07-01 00:00:00|     16

In [14]:
# Load weekly KPI aggregates into Cassandra demand_zones for Grafana
kpi_for_cassandra = (
    kpi_trips_per_zone
    .withColumn("city", F.lit("Casablanca"))
    .withColumn("window_start", (F.unix_timestamp("week_start") * 1000).cast("long"))
    .withColumn("active_vehicles", F.col("unique_taxis").cast("int"))
    .withColumn("pending_requests", F.col("total_trips").cast("int"))
    .withColumn(
        "ratio",
        (F.col("total_trips") / F.greatest(F.col("unique_taxis"), F.lit(1))).cast("double")
    )
    .select(
        "city", "zone_id", "window_start",
        "active_vehicles", "pending_requests", "ratio"
    )
)

(
    kpi_for_cassandra.write
    .format("org.apache.spark.sql.cassandra")
    .options(table="demand_zones", keyspace="taasim")
    .mode("append")
    .save()
)

print("✅ Weekly KPI aggregates written to Cassandra demand_zones.")

✅ Weekly KPI aggregates written to Cassandra demand_zones.
